# Комплексный анализ влияния ML-подходов на предсказания + Ансамблирование лучших моделей

Этот ноутбук:
1. **Часть 1** — Сводный анализ всех этапов (baseline → tuning → clustering → LLM-фичи)
2. **Часть 2** — Визуализации: сравнение подходов, дельты, распределения ошибок
3. **Часть 3** — Ансамблирование лучших моделей (Simple Average, Weighted Average, Stacking, Blending, Voting)
4. **Часть 4** — Финальное сравнение ансамблей с одиночными моделями

---

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import json, time, copy

from sklearn.linear_model import (
    Lasso, Ridge, ElasticNet, HuberRegressor, BayesianRidge
)
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    GradientBoostingRegressor, StackingRegressor, VotingRegressor
)
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, KFold, cross_val_predict, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.base import clone

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 140, "font.size": 10})

seed = 2
DURATION_CAP = 960
target_col = "duration_hours"

print("Imports OK")

## 0. Загрузка данных и артефактов

In [ ]:
# === Data ===
DATA_CANDIDATES = [
    Path("./data_finall_without_TTM.csv"),
    Path("/mnt/data/data_finall_without_TTM.csv"),
]
DATA_PATH = None
for c in DATA_CANDIDATES:
    if c.exists():
        DATA_PATH = c
        break
if DATA_PATH is None:
    raise FileNotFoundError("data_finall_without_TTM.csv not found")

df = pd.read_csv(DATA_PATH)
split_idx = int(len(df) * 0.8)

train_full = df.iloc[:split_idx].copy()
test_full  = df.iloc[split_idx:].copy()

train_filt   = train_full[train_full[target_col] < DURATION_CAP].copy()
test_typical = test_full[test_full[target_col] < DURATION_CAP].copy()

Xtrain = train_filt.drop(columns=[target_col])
ytrain = train_filt[target_col]

Xtest_typ  = test_typical.drop(columns=[target_col])
ytest_typ  = test_typical[target_col]

Xtest_full = test_full.drop(columns=[target_col])
ytest_full = test_full[target_col]

print(f"Train filtered: {len(train_filt)}, Test typical: {len(test_typical)}, Test full: {len(test_full)}")
print(f"Features: {Xtrain.shape[1]}")

In [ ]:
# === Загрузка артефактов ===
baseline_holdout = pd.read_csv("./artifacts_baseline_pointfix/primary_holdout_summary.csv")
baseline_cv      = pd.read_csv("./artifacts_baseline_pointfix/cv_summary.csv")
tuned_summary    = pd.read_csv("./artifacts_tuning_pointfix/final_summary.csv")
cluster_best     = pd.read_csv("./artifacts_meta_final/cluster_best_by_model.csv")

# LLM results — 3 провайдера
LLM_RESULT_PATHS = {
    "GPT":    Path("./artifacts/llm_cluster_compare_results.csv"),
    "Claude": Path("./claude_api/artifacts_claude_api/llm_cluster_compare_results.csv"),
    "Ollama": Path("./ollama_local/artifacts_ollama_local/llm_cluster_compare_results.csv"),
}

provider_dfs = {}
for provider, path in LLM_RESULT_PATHS.items():
    if path.exists():
        df_p = pd.read_csv(path)
        df_p["llm_provider"] = provider
        provider_dfs[provider] = df_p
        print(f"[OK] {provider}: {len(df_p)} строк, эксперименты: {sorted(df_p['experiment'].unique())}")
    else:
        print(f"[SKIP] {provider}: файл не найден")

# Мастер-таблица: все 90 строк (3 провайдера × 3 эксперимента × 10 моделей)
master_df = pd.concat(provider_dfs.values(), ignore_index=True)
print(f"\nМастер-таблица: {len(master_df)} строк")
print(f"Провайдеры: {sorted(master_df['llm_provider'].unique())}")
print(f"Модели: {sorted(master_df['model'].unique())}")
print(f"Эксперименты: {sorted(master_df['experiment'].unique())}")


---
# Часть 1. Полное сравнение LLM-подходов

**3 провайдера × 3 эксперимента × 10 моделей = 90 конфигураций**


In [ ]:
# === 1.1 Мастер-таблица: 90 конфигураций ===

display_cols = ["llm_provider","experiment","model",
                "holdout_typical_MAE","holdout_full_MAE","delta_typical","delta_full",
                "RMSE_typical","MdAE_typical","cv_MAE_internal"]

master_sorted = master_df[display_cols].sort_values(
    ["holdout_typical_MAE","holdout_full_MAE"]).reset_index(drop=True)
master_sorted.insert(0, "rank", range(1, len(master_sorted)+1))

print("=" * 110)
print("ПОЛНАЯ ТАБЛИЦА: 3 провайдера × 3 эксперимента × 10 моделей")
print("=" * 110)
display(master_sorted.round(2))

# Лучшее по каждой модели
best_per_model = (
    master_df.sort_values("holdout_typical_MAE")
    .groupby("model").head(1)
    .reset_index(drop=True)
    [["model","llm_provider","experiment","holdout_typical_MAE","holdout_full_MAE","delta_typical"]]
    .sort_values("holdout_typical_MAE")
)
print("\nЛучшая конфигурация для каждой модели:")
display(best_per_model.round(2))


In [ ]:
# === 1.2 Heatmap: model × experiment для каждого провайдера ===

providers_available = list(provider_dfs.keys())
n_p = len(providers_available)
fig, axes = plt.subplots(1, n_p, figsize=(11 * n_p, 10))
if n_p == 1: axes = [axes]

for ax, prov in zip(axes, providers_available):
    pivot = provider_dfs[prov].pivot_table(
        index="model", columns="experiment", values="holdout_typical_MAE"
    )
    pivot = pivot.sort_values(pivot.min(axis=1).name if len(pivot.columns)==1 else pivot.columns[0])
    pivot = pivot.sort_values(by=list(pivot.columns), key=lambda s: s.rank())
    # sort rows by best across experiments
    pivot["_best"] = pivot.min(axis=1)
    pivot = pivot.sort_values("_best").drop(columns="_best")
    sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd_r",
                linewidths=0.5, ax=ax, cbar_kws={"label": "MAE_typical"})
    ax.set_title(f"LLM-сценарии — {prov}\nmodel × experiment (MAE_typical)",
                fontsize=12, fontweight="bold")
    ax.set_xlabel("Эксперимент")
    ax.set_ylabel("Модель")

plt.suptitle("MAE по LLM-сценариям: сравнение всех провайдеров", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# === 1.3 Cross-provider: один провайдер vs другой (по experiment+model) ===

# Pivot: строки = (model, experiment), столбцы = провайдер
cross = master_df.pivot_table(
    index=["model","experiment"], columns="llm_provider",
    values="holdout_typical_MAE"
).round(2)
cross["best_provider"] = cross.idxmin(axis=1)
cross["spread"] = cross.drop(columns="best_provider").max(axis=1) - cross.drop(columns="best_provider").min(axis=1)
cross = cross.sort_values("spread", ascending=False)

print("Cross-provider сравнение (MAE_typical), отсортировано по разбросу:")
display(cross)

# Win counts
wins = cross["best_provider"].value_counts()
print("\nПобед (лучший MAE_typical) по конфигурациям model×experiment:")
for p, w in wins.items():
    print(f"  {p}: {w} из {len(cross)}")


In [ ]:
# === 1.4 Grouped bar: MAE_typical по провайдерам для каждого эксперимента ===

experiments = sorted(master_df["experiment"].unique())
fig, axes = plt.subplots(1, len(experiments), figsize=(8*len(experiments), 8), sharey=False)
if len(experiments) == 1: axes = [axes]

cmap_p = {"GPT": "#636EFA", "Claude": "#EF553B", "Ollama": "#00CC96"}
models_order = sorted(master_df["model"].unique())

for ax, exp in zip(axes, experiments):
    sub = master_df[master_df["experiment"]==exp].set_index("model")
    x = np.arange(len(models_order))
    bar_w = 0.25
    for i, (prov, color) in enumerate(cmap_p.items()):
        if prov not in sub["llm_provider"].values: continue
        sub_p = master_df[(master_df["experiment"]==exp) & (master_df["llm_provider"]==prov)].set_index("model")
        vals = [sub_p.loc[m,"holdout_typical_MAE"] if m in sub_p.index else np.nan for m in models_order]
        ax.bar(x + i*bar_w, vals, bar_w, label=prov, color=color, alpha=0.85, edgecolor="white")
    ax.set_xticks(x + bar_w)
    ax.set_xticklabels(models_order, rotation=45, ha="right")
    ax.set_title(f"Эксперимент: {exp}", fontweight="bold")
    ax.set_ylabel("MAE_typical")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("MAE_typical: сравнение провайдеров по каждому эксперименту и модели",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# === 1.5 Delta: улучшение LLM-фич vs baseline для каждого провайдера ===

fig, axes = plt.subplots(1, len(providers_available), figsize=(8*len(providers_available), 7), sharey=True)
if len(providers_available) == 1: axes = [axes]

for ax, prov in zip(axes, providers_available):
    # лучший delta_typical для каждой модели этого провайдера
    best_delta = (
        provider_dfs[prov]
        .sort_values("delta_typical", ascending=False)
        .groupby("model").head(1)
        .set_index("model")
    )
    deltas = best_delta["delta_typical"].sort_values(ascending=False)
    colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in deltas]
    ax.barh(deltas.index, deltas.values, color=colors, edgecolor="white")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{prov}\nΔMAE_typical (лучший эксперимент для каждой модели)", fontweight="bold")
    ax.set_xlabel("ΔMAE (+ = лучше baseline)")
    ax.grid(axis="x", alpha=0.3)
    for i, v in enumerate(deltas.values):
        ax.text(v + 0.1, i, f"{v:+.2f}", va="center", fontsize=9)

plt.suptitle("Улучшение MAE от LLM-фич vs baseline (лучший вариант каждой модели)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# === 1.6 Scatter: MAE_typical vs MAE_full — все 90 конфигураций ===

fig, ax = plt.subplots(figsize=(13, 9))
markers = {"GPT": "o", "Claude": "s", "Ollama": "^"}
exp_colors = {"llm_only": "#636EFA", "cluster_before_llm": "#EF553B", "llm_then_cluster": "#00CC96"}

for _, row in master_df.iterrows():
    ax.scatter(row["holdout_typical_MAE"], row["holdout_full_MAE"],
              marker=markers.get(row["llm_provider"], "o"),
              color=exp_colors.get(row["experiment"], "gray"),
              s=80, alpha=0.75, edgecolors="black", linewidths=0.4)

# Легенда
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
leg1 = [Line2D([0],[0], marker=m, color="w", markerfacecolor="gray",
               markersize=10, markeredgecolor="black", label=p)
        for p, m in markers.items()]
leg2 = [Patch(facecolor=c, label=e) for e, c in exp_colors.items()]
l1 = ax.legend(handles=leg1, title="Провайдер", loc="upper left", fontsize=9)
ax.add_artist(l1)
ax.legend(handles=leg2, title="Эксперимент", loc="upper right", fontsize=9)

ax.set_xlabel("MAE_typical (< 960h)")
ax.set_ylabel("MAE_full (все задачи)")
ax.set_title("Все 90 конфигураций: MAE_typical vs MAE_full\n"
             "(форма = провайдер, цвет = эксперимент)", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
# Часть 2. Сравнение ML-этапов (Baseline → Tuning → Clustering → LLM)


In [ ]:
# === 2.1 Сводная таблица по этапам ===

NAME_MAP = {
    "Gboost-Reg":"GBoost","XGB_reg":"XGB","Hub-Reg":"HuberReg",
    "Elastic":"ElasticNet","RF_reg":"RF","ET_reg":"ExtraTrees",
    "BR_reg":"Bagging","DT_reg":"DT","KNN_reg":"KNN",
    "RFR_PCA":"RFR_PCA","XGBR_PCA":"XGBR_PCA","SVR":"SVR",
    "BayRidge":"BayRidge","Lasso":"Lasso","Ridge":"Ridge"
}

stage_base = baseline_holdout[["Model","MAE_typical"]].copy()
stage_base["Model"] = stage_base["Model"].str.replace("Scaled_","",regex=False).map(lambda x: NAME_MAP.get(x,x))
stage_base = stage_base.rename(columns={"Model":"model","MAE_typical":"mae_baseline"})

stage_tuned = tuned_summary[["model","holdout_typical_MAE"]].copy()
stage_tuned["model"] = stage_tuned["model"].map(lambda x: NAME_MAP.get(x,x))
stage_tuned = stage_tuned.rename(columns={"holdout_typical_MAE":"mae_tuned"})

stage_cluster = cluster_best[["model","mae_typical"]].rename(columns={"mae_typical":"mae_cluster"})

# Лучшее LLM для каждой модели (across providers+experiments)
stage_llm = (
    master_df.sort_values("holdout_typical_MAE")
    .groupby("model").head(1)
    .reset_index(drop=True)
    [["model","holdout_typical_MAE","llm_provider","experiment"]]
    .rename(columns={"holdout_typical_MAE":"mae_llm_best",
                     "llm_provider":"best_provider","experiment":"best_exp"})
)

summary = stage_base.merge(stage_tuned, on="model", how="left")
summary = summary.merge(stage_cluster, on="model", how="left")
summary = summary.merge(stage_llm, on="model", how="left")
summary["delta_tuning"]  = summary["mae_baseline"] - summary["mae_tuned"]
summary["delta_cluster"] = summary["mae_baseline"] - summary["mae_cluster"]
summary["delta_llm"]     = summary["mae_baseline"] - summary["mae_llm_best"]

display(summary[["model","mae_baseline","mae_tuned","delta_tuning",
                  "mae_cluster","delta_cluster","mae_llm_best","delta_llm",
                  "best_provider","best_exp"]].sort_values("mae_baseline").round(2))


In [ ]:
# === 2.2 Grouped bar: этапы ML-пайплайна ===

plot_df = summary.dropna(subset=["mae_baseline","mae_tuned","mae_llm_best"]).sort_values("mae_baseline").head(10)
x = np.arange(len(plot_df))
w = 0.22

fig, ax = plt.subplots(figsize=(16, 7))
ax.bar(x - 1.5*w, plot_df["mae_baseline"], w, label="Baseline",     color="#636EFA", edgecolor="white")
ax.bar(x - 0.5*w, plot_df["mae_tuned"].fillna(0), w, label="Tuned", color="#EF553B", edgecolor="white")
if "mae_cluster" in plot_df:
    ax.bar(x + 0.5*w, plot_df["mae_cluster"].fillna(0), w, label="+ Clustering", color="#00CC96", edgecolor="white")
ax.bar(x + 1.5*w, plot_df["mae_llm_best"], w, label="+ LLM (лучший)", color="#AB63FA", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(plot_df["model"], rotation=45, ha="right")
ax.set_ylabel("MAE_typical")
ax.set_title("Влияние ML-этапов на качество предсказаний", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# === 2.3 Линейный прогресс по этапам ===

focus_models = ["XGB","GBoost","Lasso","BayRidge","ElasticNet","HuberReg","Ridge"]
stages = ["Baseline","Tuned","+ Clustering","+ LLM (best)"]
col_keys = ["mae_baseline","mae_tuned","mae_cluster","mae_llm_best"]

fig, ax = plt.subplots(figsize=(13, 7))
cmap = plt.cm.tab10

for idx, m in enumerate(focus_models):
    row = summary[summary["model"]==m]
    if row.empty: continue
    r = row.iloc[0]
    vals = [r.get(k, np.nan) for k in col_keys]
    valid_s = [s for s,v in zip(stages,vals) if pd.notna(v)]
    valid_v = [v for v in vals if pd.notna(v)]
    ax.plot(valid_s, valid_v, marker="o", linewidth=2, markersize=8,
            label=m, color=cmap(idx))
    ax.annotate(f"{valid_v[-1]:.1f}", (valid_s[-1], valid_v[-1]),
               textcoords="offset points", xytext=(8,0), fontsize=8)

ax.set_ylabel("MAE_typical")
ax.set_title("Прогресс качества по этапам ML-пайплайна", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
# Часть 3. Ансамблирование: лучшая конфигурация каждой модели

Для каждой из 10 моделей берём **лучший (провайдер, эксперимент)** по `holdout_typical_MAE`.
Загружаем соответствующий feature-файл, обучаем модель с точными `best_params` из артефактов.
Затем ансамблируем 10 лучших моделей несколькими способами.

ℹ️ Для эксперимента `llm_then_cluster` feature-файлы не сохраняются (кластеризация строится
на лету), поэтому используется следующий лучший вариант с доступным файлом.


In [ ]:
# === 3.0 Регистрация feature-файлов ===

LLM_FEAT_COLS = [
    "execution_complexity_1_5","coordination_risk_1_5",
    "testing_effort_1_5","uncertainty_1_5",
    "urgent_delivery_0_1","likely_long_task_0_1",
]

# (provider, experiment) -> (train_path, test_path)  — только там, где есть файлы
FEAT_FILE_REGISTRY = {
    ("GPT",    "llm_only"):           (Path("./artifacts/llm_features_llm_only_train.csv"),
                                       Path("./artifacts/llm_features_llm_only_test.csv")),
    ("GPT",    "cluster_before_llm"): (Path("./artifacts/llm_features_cluster_before_llm_train.csv"),
                                       Path("./artifacts/llm_features_cluster_before_llm_test.csv")),
    ("Claude", "llm_only"):           (Path("./claude_api/artifacts_claude_api/llm_features_claude_api_claude_sonnet_4_6_llm_only_train.csv"),
                                       Path("./claude_api/artifacts_claude_api/llm_features_claude_api_claude_sonnet_4_6_llm_only_test.csv")),
    ("Claude", "cluster_before_llm"): (Path("./claude_api/artifacts_claude_api/llm_features_claude_api_claude_sonnet_4_6_cluster_before_llm_train.csv"),
                                       Path("./claude_api/artifacts_claude_api/llm_features_claude_api_claude_sonnet_4_6_cluster_before_llm_test.csv")),
    ("Ollama", "llm_only"):           (Path("./ollama_local/artifacts_ollama_local/llm_features_ollama_llama3_2_3b_llm_only_train.csv"),
                                       Path("./ollama_local/artifacts_ollama_local/llm_features_ollama_llama3_2_3b_llm_only_test.csv")),
    ("Ollama", "cluster_before_llm"): (Path("./ollama_local/artifacts_ollama_local/llm_features_ollama_llama3_2_3b_cluster_before_llm_train.csv"),
                                       Path("./ollama_local/artifacts_ollama_local/llm_features_ollama_llama3_2_3b_cluster_before_llm_test.csv")),
}

typ_mask = ytest_full.values < DURATION_CAP

def get_aug_data(provider, experiment):
    """Возвращает (Xtr_aug, Xte_full_aug, Xte_typ_aug) или None если файл недоступен."""
    key = (provider, experiment)
    if key not in FEAT_FILE_REGISTRY:
        return None
    tr_path, te_path = FEAT_FILE_REGISTRY[key]
    if not tr_path.exists() or not te_path.exists():
        return None
    prefix = f"{provider}_{experiment}_"
    llm_tr = pd.read_csv(tr_path)[LLM_FEAT_COLS].reset_index(drop=True).add_prefix(prefix)
    llm_te = pd.read_csv(te_path)[LLM_FEAT_COLS].reset_index(drop=True).add_prefix(prefix)
    Xtr_aug  = pd.concat([Xtrain.reset_index(drop=True), llm_tr], axis=1)
    Xte_full_aug = pd.concat([Xtest_full.reset_index(drop=True), llm_te], axis=1)
    Xte_typ_aug  = Xte_full_aug[typ_mask].reset_index(drop=True)
    return Xtr_aug, Xte_full_aug, Xte_typ_aug

print("Проверка доступности feature-файлов:")
for key in FEAT_FILE_REGISTRY:
    data = get_aug_data(*key)
    status = "OK" if data else "MISS"
    print(f"  [{status}] {key[0]:8s} / {key[1]}")


In [ ]:
# === 3.1 Выбираем лучшую доступную конфигурацию для каждой модели ===

from sklearn.linear_model import Lasso, Ridge, ElasticNet, HuberRegressor, BayesianRidge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

MODEL_CLASS_MAP = {
    "XGB":       XGBRegressor,
    "GBoost":    GradientBoostingRegressor,
    "HuberReg":  HuberRegressor,
    "Lasso":     Lasso,
    "ElasticNet": ElasticNet,
    "BayRidge":  BayesianRidge,
    "Ridge":     Ridge,
    "KNN":       KNeighborsRegressor,
    "RF":        RandomForestRegressor,
    "ExtraTrees": ExtraTreesRegressor,
}

# Для каждой модели ищем лучший (provider, experiment) среди тех, для которых есть файл
available_keys = set(FEAT_FILE_REGISTRY.keys())

best_configs = []  # list of dicts

for model_name in sorted(master_df["model"].unique()):
    # Отбираем строки этой модели, у которых есть feature-файл
    candidates = master_df[
        (master_df["model"] == model_name) &
        (master_df.apply(lambda r: (r["llm_provider"], r["experiment"]) in available_keys, axis=1))
    ].sort_values("holdout_full_MAE")

    if candidates.empty:
        print(f"[SKIP] {model_name}: нет доступных feature-файлов")
        continue

    best_row = candidates.iloc[0]
    best_configs.append({
        "model_name": model_name,
        "provider":   best_row["llm_provider"],
        "experiment": best_row["experiment"],
        "mae_typical_ref": best_row["holdout_typical_MAE"],
        "mae_full_ref":    best_row["holdout_full_MAE"],
        "best_params_json": best_row["best_params"],
    })
    _prov = best_row["llm_provider"]; _exp = best_row["experiment"]; _mtyp = best_row["holdout_typical_MAE"]; _mfull = best_row["holdout_full_MAE"]
    print(f"{model_name:<12s}: {_prov:<8s} / {_exp:<22s} MAE_typ={_mtyp:.2f}, MAE_full={_mfull:.2f}")

print(f"\nИтого конфигураций для ансамбля: {len(best_configs)}")


In [ ]:
# === 3.2 Обучаем каждую модель с её best_params на её best feature-set ===

def build_pipeline(model_name, params_json):
    """Создаёт Pipeline(StandardScaler + model) с best_params из JSON."""
    import inspect
    params = json.loads(params_json)
    ModelClass = MODEL_CLASS_MAP[model_name]
    sig = inspect.signature(ModelClass.__init__)
    # Если конструктор принимает **kwargs (XGBRegressor) — передаём всё, иначе фильтруем
    uses_kwargs = any(
        p.kind == inspect.Parameter.VAR_KEYWORD
        for p in sig.parameters.values()
    )
    if uses_kwargs:
        clean = {k: v for k, v in params.items() if v is not None}
    else:
        valid_keys = set(sig.parameters.keys()) - {"self"}
        clean = {k: v for k, v in params.items() if k in valid_keys and v is not None}
    if model_name == "XGB":
        clean["verbosity"] = 0
        clean["n_jobs"] = -1
    elif model_name in ("RF", "ExtraTrees"):
        clean["n_jobs"] = -1
    model = ModelClass(**clean)
    return Pipeline([("sc", StandardScaler()), ("m", model)])

def reg_metrics(y_true, y_pred):
    return {
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MdAE": median_absolute_error(y_true, y_pred),
    }

all_preds_typ  = {}
all_preds_full = {}
solo_scores    = []

print("{:<12s} {:<8s} {:<24s} {:>11s} {:>11s} {:>12s}".format("Модель","Провайдер","Эксперимент","MAE_typ_ref","MAE_typ_now","MAE_full_now"))
print("-" * 82)

for cfg in best_configs:
    mname   = cfg["model_name"]
    prov    = cfg["provider"]
    exp     = cfg["experiment"]
    key     = f"{mname}\n({prov}/{exp[:3]})"
    key_short = f"{mname}_{prov[:3]}_{exp[:3]}"

    aug_data = get_aug_data(prov, exp)
    if aug_data is None:
        print(f"[SKIP] {mname}: нет данных для ({prov}, {exp})")
        continue
    Xtr_aug, Xte_full_aug, Xte_typ_aug = aug_data

    pipe = build_pipeline(mname, cfg["best_params_json"])
    pipe.fit(Xtr_aug, ytrain)

    p_typ  = pipe.predict(Xte_typ_aug)
    p_full = pipe.predict(Xte_full_aug)

    all_preds_typ[key_short]  = p_typ
    all_preds_full[key_short] = p_full

    m_t = reg_metrics(ytest_typ, p_typ)
    m_f = reg_metrics(ytest_full, p_full)
    solo_scores.append({"model": key_short, "label": f"{mname} ({prov}/{exp[:7]})",
                        "source": "individual",
                        "MAE_typ": m_t["MAE"], "RMSE_typ": m_t["RMSE"],
                        "MdAE_typ": m_t["MdAE"], "MAE_full": m_f["MAE"]})

    _ref = cfg["mae_typical_ref"]; _mt = m_t["MAE"]; _mf = m_f["MAE"]
    print(f"{mname:<12s} {prov:<8s} {exp:<24s} {_ref:>11.2f} {_mt:>11.2f} {_mf:>12.2f}")

solo_df = pd.DataFrame(solo_scores).sort_values("MAE_full")
best_solo_full = solo_df.iloc[0]["MAE_full"]
best_solo_typ  = solo_df.iloc[0]["MAE_typ"]
print(f"\nЛучший соло: {solo_df.iloc[0]['label']} — MAE_typ={best_solo_typ:.2f}, MAE_full={best_solo_full:.2f}")

# === Дополнительно: обучаем ВСЕ доступные конфиги сильных моделей (XGB + GBoost) ===
# для диверсифицированного ансамбля (разные провайдеры = разные LLM-фичи)
print("\n--- Все доступные конфиги XGB и GBoost ---")
all_xgb_preds_typ  = {}
all_xgb_preds_full = {}
xgb_scores = []

for _, row in master_df[master_df["model"].isin(["XGB","GBoost"])].iterrows():
    prov = row["llm_provider"]; exp = row["experiment"]
    if (prov, exp) not in FEAT_FILE_REGISTRY: continue
    key = f"{row['model']}_{prov[:3]}_{exp[:3]}"
    aug_data = get_aug_data(prov, exp)
    if aug_data is None: continue
    Xtr_a, Xte_full_a, Xte_typ_a = aug_data
    # Заполняем NaN медианой (GBoost не принимает NaN нативно)
    med = Xtr_a.median(numeric_only=True)
    Xtr_a      = Xtr_a.fillna(med)
    Xte_full_a = Xte_full_a.fillna(med)
    Xte_typ_a  = Xte_typ_a.fillna(med)
    pipe = build_pipeline(row["model"], row["best_params"])
    pipe.fit(Xtr_a, ytrain)
    p_typ  = pipe.predict(Xte_typ_a)
    p_full = pipe.predict(Xte_full_a)
    all_xgb_preds_typ[key]  = p_typ
    all_xgb_preds_full[key] = p_full
    m_t = reg_metrics(ytest_typ, p_typ)
    m_f = reg_metrics(ytest_full, p_full)
    xgb_scores.append({"key": key, "MAE_typ": m_t["MAE"], "MAE_full": m_f["MAE"]})
    _mtyp = m_t["MAE"]; _mfull = m_f["MAE"]
    print(f"  {key:<35s} MAE_typ={_mtyp:.2f}, MAE_full={_mfull:.2f}")

xgb_score_df = pd.DataFrame(xgb_scores).sort_values("MAE_full")
print(f"\nВсего XGB+GBoost конфигов: {len(xgb_scores)}")


In [ ]:
# === 3.3 Все виды ансамблирования ===

ens_results = []

def add_ens(name, p_typ, p_full):
    m_t = reg_metrics(ytest_typ, p_typ)
    m_f = reg_metrics(ytest_full, p_full)
    ens_results.append({"model": name, "label": name, "source": "ensemble",
                        "MAE_typ": m_t["MAE"], "RMSE_typ": m_t["RMSE"],
                        "MdAE_typ": m_t["MdAE"], "MAE_full": m_f["MAE"]})
    return m_t["MAE"], m_f["MAE"]

keys_all   = list(all_preds_typ.keys())
keys_xgb_all   = list(all_xgb_preds_typ.keys())   # все XGB+GBoost конфиги
keys_xgb_only  = [k for k in keys_xgb_all if k.startswith("XGB_")]
keys_boost_all = keys_xgb_all  # и XGB и GBoost

# ---- A. XGB × all providers (основной ансамбль) ----
if keys_xgb_only:
    mae_t, mae_f = add_ens(f"Avg XGB all providers ({len(keys_xgb_only)})",
        np.mean([all_xgb_preds_typ[k] for k in keys_xgb_only], axis=0),
        np.mean([all_xgb_preds_full[k] for k in keys_xgb_only], axis=0))
    print(f"Avg XGB all providers ({len(keys_xgb_only)}):   MAE_typ={mae_t:.2f}, MAE_full={mae_f:.2f}")

    wx = {k: 1.0/reg_metrics(ytest_full, all_xgb_preds_full[k])["MAE"]**2 for k in keys_xgb_only}
    wx_s = sum(wx.values()); wx = {k: v/wx_s for k,v in wx.items()}
    mae_t, mae_f = add_ens("WeightedAvg XGB (inv MAE²)",
        sum(wx[k]*all_xgb_preds_typ[k] for k in keys_xgb_only),
        sum(wx[k]*all_xgb_preds_full[k] for k in keys_xgb_only))
    print(f"WeightedAvg XGB (inv MAE²):     MAE_typ={mae_t:.2f}, MAE_full={mae_f:.2f}")

    mae_t, mae_f = add_ens("Median XGB all providers",
        np.median([all_xgb_preds_typ[k] for k in keys_xgb_only], axis=0),
        np.median([all_xgb_preds_full[k] for k in keys_xgb_only], axis=0))
    print(f"Median XGB all providers:       MAE_typ={mae_t:.2f}, MAE_full={mae_f:.2f}")

# ---- B. XGB + GBoost (все конфиги) ----
if keys_boost_all:
    mae_t, mae_f = add_ens(f"Avg XGB+GBoost all ({len(keys_boost_all)})",
        np.mean([all_xgb_preds_typ[k] for k in keys_boost_all], axis=0),
        np.mean([all_xgb_preds_full[k] for k in keys_boost_all], axis=0))
    print(f"Avg XGB+GBoost all ({len(keys_boost_all)}):      MAE_typ={mae_t:.2f}, MAE_full={mae_f:.2f}")

    wb = {k: 1.0/reg_metrics(ytest_full, all_xgb_preds_full[k])["MAE"]**2 for k in keys_boost_all}
    wb_s = sum(wb.values()); wb = {k: v/wb_s for k,v in wb.items()}
    mae_t, mae_f = add_ens("WeightedAvg XGB+GBoost (inv MAE²)",
        sum(wb[k]*all_xgb_preds_typ[k] for k in keys_boost_all),
        sum(wb[k]*all_xgb_preds_full[k] for k in keys_boost_all))
    print(f"WeightedAvg XGB+GBoost:         MAE_typ={mae_t:.2f}, MAE_full={mae_f:.2f}")

# ---- C. Avg ALL (10 лучших — для справки) ----
mae_t, mae_f = add_ens("Avg ALL 10 моделей (справка)",
    np.mean([all_preds_typ[k] for k in keys_all], axis=0),
    np.mean([all_preds_full[k] for k in keys_all], axis=0))
print(f"\nАvg ALL 10 models (ref):       MAE_typ={mae_t:.2f}, MAE_full={mae_f:.2f}")

if len(keys_all) >= 3:
    top3_keys = solo_df.head(3)["model"].tolist()
    mae_t, mae_f = add_ens("Avg top-3 best (10 моделей)",
        np.mean([all_preds_typ[k] for k in top3_keys], axis=0),
        np.mean([all_preds_full[k] for k in top3_keys], axis=0))
    print(f"Avg top-3 (10 models):         MAE_typ={mae_t:.2f}, MAE_full={mae_f:.2f}")

# ---- D. Stacking (KFold Ridge meta) ----
print("\nStacking (KFold Ridge meta)...")
t0 = time.time()

# Строим общий X: базовые фичи + LLM-колонки всех провайдеров (с суффиксом провайдера)
aug_cache = {}
for cfg in best_configs:
    key = (cfg["provider"], cfg["experiment"])
    if key not in aug_cache:
        d = get_aug_data(cfg["provider"], cfg["experiment"])
        if d: aug_cache[key] = d

def build_unified(base_df, slot):
    """slot=0 → Xtr, 1 → Xte_full, 2 → Xte_typ"""
    result = base_df.reset_index(drop=True).copy()
    for (prov, exp), triple in aug_cache.items():
        aug = triple[slot].reset_index(drop=True)
        llm_cols = [c for c in aug.columns if c not in base_df.columns]
        suffix = f"_{prov[:3]}_{exp[:3]}"
        renamed = aug[llm_cols].rename(columns={c: c+suffix for c in llm_cols})
        result = pd.concat([result, renamed], axis=1)
    # fill NaN with column median (handles failed LLM calls)
    result = result.fillna(result.median(numeric_only=True))
    return result

Xtr_all      = build_unified(Xtrain,    0)
Xte_full_all = build_unified(Xtest_full, 1)
Xte_typ_all  = build_unified(Xtest_typ,  2)

stacking_ests = []
for cfg in best_configs:
    mname = cfg["model_name"]
    key_short = f"{mname}_{cfg['provider'][:3]}_{cfg['experiment'][:3]}"
    pipe = build_pipeline(mname, cfg["best_params_json"])
    stacking_ests.append((key_short, pipe))

stacking_ridge = StackingRegressor(
    estimators=stacking_ests,
    final_estimator=Ridge(alpha=10.0),
    cv=KFold(n_splits=5, shuffle=False),
    n_jobs=-1, passthrough=False,
)
stacking_ridge.fit(Xtr_all, ytrain)
mae_t, mae_f = add_ens("Stacking (Ridge meta, 10 best)",
    stacking_ridge.predict(Xte_typ_all),
    stacking_ridge.predict(Xte_full_all))
print(f"Stacking (Ridge):              MAE_typ={mae_t:.2f}, MAE_full={mae_f:.2f}  [{time.time()-t0:.1f}s]")

# ---- E. Blending (OOF → Huber meta, только XGB+GBoost) ----
# HuberRegressor ~ MAE-оптимальный мета-ученик (в отличие от Ridge/MSE)
# Используем ВСЕ XGB+GBoost конфиги (6-12 шт.) для максимальной диверсификации
print("\nBlending (OOF → Huber meta, все XGB+GBoost)...")
t0 = time.time()

# Собираем все конфиги XGB+GBoost из master_df (все доступные)
blend_configs = []
for _, row in master_df[master_df["model"].isin(["XGB","GBoost"])].iterrows():
    prov = row["llm_provider"]; exp = row["experiment"]
    if (prov, exp) not in FEAT_FILE_REGISTRY: continue
    blend_configs.append({
        "model_name": row["model"],
        "provider": prov,
        "experiment": exp,
        "best_params_json": row["best_params"],
        "key": f"{row['model']}_{prov[:3]}_{exp[:3]}",
    })

tscv_blend = TimeSeriesSplit(n_splits=5)  # больше фолдов → больше OOF-данных
n_b = len(blend_configs)
oof_b     = np.zeros((len(Xtrain), n_b))
te_typ_b  = np.zeros((len(Xtest_typ), n_b))
te_full_b = np.zeros((len(Xtest_full), n_b))

for col_i, cfg in enumerate(blend_configs):
    aug_data = get_aug_data(cfg["provider"], cfg["experiment"])
    if aug_data is None: continue
    Xtr_a, Xte_full_a, Xte_typ_a = aug_data
    med = Xtr_a.median(numeric_only=True)
    Xtr_a = Xtr_a.fillna(med); Xte_full_a = Xte_full_a.fillna(med); Xte_typ_a = Xte_typ_a.fillna(med)
    pipe = build_pipeline(cfg["model_name"], cfg["best_params_json"])
    fold_te_typ, fold_te_full = [], []
    for tr_idx, va_idx in tscv_blend.split(Xtr_a):
        est = clone(pipe)
        est.fit(Xtr_a.iloc[tr_idx], ytrain.iloc[tr_idx])
        oof_b[va_idx, col_i] = est.predict(Xtr_a.iloc[va_idx])
        fold_te_typ.append(est.predict(Xte_typ_a))
        fold_te_full.append(est.predict(Xte_full_a))
    te_typ_b[:, col_i]  = np.mean(fold_te_typ, axis=0)
    te_full_b[:, col_i] = np.mean(fold_te_full, axis=0)

valid_rows = oof_b.sum(axis=1) != 0
blend_keys = [c["key"] for c in blend_configs]

# Мета-ученик 1: HuberRegressor (робастный, ближе к MAE)
meta_huber = HuberRegressor(epsilon=1.35, alpha=0.01, max_iter=500)
meta_huber.fit(oof_b[valid_rows], ytrain.values[valid_rows])
mae_t, mae_f = add_ens("Blending (OOF→Huber, XGB+GBoost)",
    meta_huber.predict(te_typ_b),
    meta_huber.predict(te_full_b))
print(f"Blending Huber:                MAE_typ={mae_t:.2f}, MAE_full={mae_f:.2f}  [{time.time()-t0:.1f}s]")

# Мета-ученик 2: Ridge (сравнение)
meta_ridge = Ridge(alpha=1.0)
meta_ridge.fit(oof_b[valid_rows], ytrain.values[valid_rows])
mae_t2, mae_f2 = add_ens("Blending (OOF→Ridge, XGB+GBoost)",
    meta_ridge.predict(te_typ_b),
    meta_ridge.predict(te_full_b))
print(f"Blending Ridge:                MAE_typ={mae_t2:.2f}, MAE_full={mae_f2:.2f}")

print("\nВеса мета-Huber (XGB+GBoost конфиги):")
for k, coef in zip(blend_keys, meta_huber.coef_):
    print(f"  {k:<35s}: {coef:+.4f}")


---
# Часть 4. Финальное сравнение


In [ ]:
# === 4.1 Итоговая таблица: соло-модели + ансамбли ===

all_rows = pd.concat([
    pd.DataFrame(solo_scores),
    pd.DataFrame(ens_results),
], ignore_index=True).sort_values("MAE_full").reset_index(drop=True)
all_rows.insert(0, "rank", range(1, len(all_rows)+1))
all_rows["vs_best_solo"] = (all_rows["MAE_full"] - best_solo_full).round(2)

print("=" * 100)
print("ФИНАЛЬНАЯ ТАБЛИЦА (сортировка по MAE_full)")
print("=" * 100)
display(all_rows[["rank","label","source","MAE_typ","MAE_full","RMSE_typ","MdAE_typ","vs_best_solo"]].round(2))

best_ens = all_rows[all_rows["source"]=="ensemble"].iloc[0]
delta = best_solo_full - best_ens["MAE_full"]
print(f"\nЛучший соло:     {solo_df.iloc[0]['label']:<40s} MAE_full={best_solo_full:.2f}")
print(f"Лучший ансамбль: {best_ens['label']:<40s} MAE_full={best_ens['MAE_full']:.2f}")
print(f"Улучшение:       {delta:+.2f} MAE ({delta/best_solo_full*100:+.2f}%)")


In [ ]:
# === 4.2 Bar chart: Ансамбли vs соло ===

fig, axes = plt.subplots(1, 2, figsize=(20, max(8, len(all_rows)*0.45+2)))

for ax, metric, title in zip(
    axes,
    ["MAE_full", "MAE_typ"],
    ["MAE Full holdout (все задачи, incl. >960h)", "MAE Typical holdout (< 960h)"]
):
    sd = all_rows.sort_values(metric)
    bar_c = ["#EF553B" if s=="ensemble" else "#636EFA" for s in sd["source"]]
    ax.barh(range(len(sd)), sd[metric], color=bar_c, edgecolor="white")
    for i, (val, lbl) in enumerate(zip(sd[metric], sd["label"])):
        ax.text(val+0.2, i, f"{val:.2f}", va="center", fontsize=8)
    ax.set_yticks(range(len(sd)))
    ax.set_yticklabels(sd["label"], fontsize=8)
    best_v = sd.iloc[0][metric]
    ax.axvline(best_v, color="green", linestyle="--", linewidth=1.5,
               alpha=0.7, label=f"Best: {best_v:.2f}")
    ax.set_xlabel(metric)
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(axis="x", alpha=0.3)

from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(facecolor="#636EFA", label="Individual (лучший вариант каждой модели)"),
    Patch(facecolor="#EF553B", label="Ensemble"),
], loc="lower center", ncol=2, fontsize=10, bbox_to_anchor=(0.5,-0.02))
plt.suptitle("Сравнение лучших конфигураций (10 моделей) и ансамблей",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# === 4.3 Корреляция предсказаний (diversity) ===

preds_df = pd.DataFrame({k: all_preds_full[k] for k in keys_all})
corr = preds_df.corr()

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm",
            center=0.99, linewidths=0.5, ax=ax,
            cbar_kws={"label": "Корреляция предсказаний"})
ax.set_title("Корреляция предсказаний (full holdout)\n"
             "Чем ниже — тем выше разнообразие → ансамбль эффективнее",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

mask_lower = np.tril(np.ones_like(corr, dtype=bool), k=-1)
avg_corr = corr.where(mask_lower).stack().mean()
print(f"Средняя попарная корреляция: {avg_corr:.4f}")


In [ ]:
# === 4.4 Веса Blending + Predicted vs Actual ===

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
coefs = meta_blend.coef_
bar_c = ["#2ecc71" if c>0 else "#e74c3c" for c in coefs]
ax.barh(keys_all, coefs, color=bar_c, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Веса мета-Ridge в Blending", fontweight="bold")
ax.set_xlabel("Коэффициент")
ax.grid(axis="x", alpha=0.3)

ax = axes[1]
p_solo = all_preds_full[solo_df.iloc[0]["model"]]
p_ens  = np.mean([all_preds_full[k] for k in keys_boost] if keys_boost else [all_preds_full[k] for k in keys_all], axis=0)
mae_s  = mean_absolute_error(ytest_full, p_solo)
mae_e  = mean_absolute_error(ytest_full, p_ens)
lim = min(float(ytest_full.max()), 2000)
ax.scatter(ytest_full, p_solo, alpha=0.2, s=10, c="#636EFA", edgecolors="none",
           label=f"Best solo ({mae_s:.1f})")
ax.scatter(ytest_full, p_ens,  alpha=0.2, s=10, c="#EF553B", edgecolors="none",
           label=f"Best ensemble ({mae_e:.1f})")
ax.plot([0,lim],[0,lim],"--", color="gray", linewidth=1.5)
ax.set_xlim(0,lim); ax.set_ylim(0,lim)
ax.set_title("Predicted vs Actual (full holdout)", fontweight="bold")
ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# === Сохранение ===
art_dir = Path("./artifacts_ensemble_analysis")
art_dir.mkdir(parents=True, exist_ok=True)
master_sorted.to_csv(art_dir / "llm_full_comparison_90rows.csv", index=False)
all_rows.to_csv(art_dir / "ensemble_final_comparison.csv", index=False)
summary.to_csv(art_dir / "pipeline_stages_comparison.csv", index=False)
print("Сохранено:")
for f in sorted(art_dir.glob("*.csv")):
    print(f"  {f.name}")
